# 21 — Debugging and Logging

## Objectives
- Apply systematic debugging strategies
- Use Java's logging API and SLF4J
- Read and interpret stack traces
- Use logging levels appropriately

## Logging Levels (Low → High)
```
TRACE → DEBUG → INFO → WARN → ERROR → FATAL
```
In production: use INFO or WARN
In development: use DEBUG or TRACE

In [1]:
import java.time.LocalDateTime;
import java.time.format.DateTimeFormatter;

// Custom logger (SLF4J pattern)
class AppLogger {
    private static final DateTimeFormatter FMT = DateTimeFormatter.ofPattern("HH:mm:ss.SSS");
    private final String name;
    
    AppLogger(String name) { this.name = name; }
    
    void debug(String msg) { log("DEBUG", msg); }
    void info(String msg)  { log("INFO ", msg); }
    void warn(String msg)  { log("WARN ", msg); }
    void error(String msg) { log("ERROR", msg); }
    void error(String msg, Throwable t) {
        log("ERROR", msg + " | " + t.getClass().getSimpleName() + ": " + t.getMessage());
    }
    
    private void log(String level, String msg) {
        System.out.printf("[%s] [%s] [%s] %s%n",
            LocalDateTime.now().format(FMT), level, name, msg);
    }
}

// Simulating application logs
AppLogger log = new AppLogger("OrderService");

log.info("Application started");
log.debug("Config loaded: max_orders=100, timeout=30s");
log.info("Processing order: ORD-2024-001 for customer alice@example.com");
log.debug("Validating payment method: VISA ending 1234");
log.info("Payment validated. Amount: INR 5,000");
log.warn("Delivery estimate delayed by 2 days due to high demand");
log.info("Order ORD-2024-001 confirmed. ETA: 3-5 business days");

// Error scenario
try {
    String nullValue = null;
    nullValue.length(); // NullPointerException
} catch (NullPointerException e) {
    log.error("Failed to process customer data", e);
}

log.warn("Retry queue depth: 847 (threshold: 500)");
log.info("Health check: DB=OK, Cache=OK, MQ=DEGRADED");
log.error("Connection pool exhausted! Active: 50/50");

[11:36:23.456] [INFO ] [OrderService] Application started
[11:36:23.471] [DEBUG] [OrderService] Config loaded: max_orders=100, timeout=30s
[11:36:23.486] [INFO ] [OrderService] Processing order: ORD-2024-001 for customer alice@example.com
[11:36:23.503] [DEBUG] [OrderService] Validating payment method: VISA ending 1234
[11:36:23.519] [INFO ] [OrderService] Payment validated. Amount: INR 5,000
[11:36:23.534] [WARN ] [OrderService] Delivery estimate delayed by 2 days due to high demand
[11:36:23.549] [INFO ] [OrderService] Order ORD-2024-001 confirmed. ETA: 3-5 business days
[11:36:23.565] [ERROR] [OrderService] Failed to process customer data | NullPointerException: Cannot invoke "String.length()" because "<local0>" is null
[11:36:23.581] [WARN ] [OrderService] Retry queue depth: 847 (threshold: 500)
[11:36:23.595] [INFO ] [OrderService] Health check: DB=OK, Cache=OK, MQ=DEGRADED
[11:36:23.616] [ERROR] [OrderService] Connection pool exhausted! Active: 50/50


## Interview Questions
1. What are the SLF4J log levels in order?
2. Why should you not log sensitive data (passwords, card numbers)?
3. How do you read a Java stack trace?
4. What is structured logging? Why is it better for production?

## Mini Challenge
Add appropriate logging to the `BankAccount` class: log at DEBUG for internal state changes, INFO for successful transactions, WARN for suspicious activity, ERROR for failures.

In [3]:
import java.time.LocalDateTime;
import java.time.format.DateTimeFormatter;

// -------------------- Custom Logger --------------------
class AppLogger {
    private static final DateTimeFormatter FMT =
            DateTimeFormatter.ofPattern("HH:mm:ss.SSS");

    private final String name;

    AppLogger(String name) {
        this.name = name;
    }

    void debug(String msg) { log("DEBUG", msg); }
    void info(String msg)  { log("INFO ", msg); }
    void warn(String msg)  { log("WARN ", msg); }
    void error(String msg) { log("ERROR", msg); }

    void error(String msg, Throwable t) {
        log("ERROR", msg + " | " +
                t.getClass().getSimpleName() + ": " + t.getMessage());
    }

    private void log(String level, String msg) {
        System.out.printf("[%s] [%s] [%s] %s%n",
                LocalDateTime.now().format(FMT),
                level,
                name,
                msg);
    }
}

// -------------------- BankAccount --------------------
class BankAccount {
    private final String accountNumber;
    private double balance;

    private final AppLogger log =
            new AppLogger("BankAccount");

    public BankAccount(String accountNumber, double initialBalance) {
        this.accountNumber = accountNumber;
        this.balance = initialBalance;

        log.info("Account created: " + accountNumber +
                 " with initial balance INR " + balance);
    }

    public void deposit(double amount) {
        log.debug("Deposit requested. Current balance=" +
                  balance + ", amount=" + amount);

        if (amount <= 0) {
            log.error("Deposit failed: amount must be positive");
            return;
        }

        balance += amount;

        log.debug("Balance updated after deposit: " + balance);
        log.info("Deposit successful: INR " + amount +
                 " | New balance: INR " + balance);
    }

    public void withdraw(double amount) {
        log.debug("Withdrawal requested. Current balance=" +
                  balance + ", amount=" + amount);

        if (amount <= 0) {
            log.error("Withdrawal failed: amount must be positive");
            return;
        }

        // WARN: suspiciously large withdrawal
        if (amount > balance * 0.8) {
            log.warn("Suspicious activity: withdrawal amount INR " +
                     amount + " is more than 80% of balance");
        }

        if (amount > balance) {
            log.error("Withdrawal failed: insufficient funds. Available balance=INR " +
                      balance);
            return;
        }

        balance -= amount;

        log.debug("Balance updated after withdrawal: " + balance);
        log.info("Withdrawal successful: INR " + amount +
                 " | Remaining balance: INR " + balance);
    }

    public void transfer(BankAccount target, double amount) {
        log.debug("Transfer initiated from " + accountNumber +
                  " to " + target.accountNumber +
                  " amount=INR " + amount);

        if (amount <= 0) {
            log.error("Transfer failed: invalid amount");
            return;
        }

        if (amount > balance) {
            log.error("Transfer failed: insufficient funds");
            return;
        }

        this.balance -= amount;
        target.balance += amount;

        log.debug("Source balance after transfer: " + this.balance);
        log.debug("Target balance after transfer: " + target.balance);

        log.info("Transfer successful: INR " + amount +
                 " transferred to account " + target.accountNumber);
    }

    public double getBalance() {
        log.debug("Balance inquiry. Current balance=" + balance);
        return balance;
    }
}

// -------------------- Demo --------------------
public class Main {
    public static void main(String[] args) {

        BankAccount alice = new BankAccount("ACC1001", 10000);
        BankAccount bob   = new BankAccount("ACC2001", 2000);

        alice.deposit(5000);        // INFO + DEBUG
        alice.withdraw(12000);      // WARN + INFO
        alice.withdraw(50000);      // WARN + ERROR
        alice.deposit(-100);        // ERROR
        alice.transfer(bob, 2000);  // INFO + DEBUG

        System.out.println("Alice balance: " + alice.getBalance());
        System.out.println("Bob balance: " + bob.getBalance());
    }
}

In [4]:
BankAccount account = new BankAccount("RONI", 5000);

account.deposit(1000);
account.withdraw(2000);
account.withdraw(3500);   // May trigger WARN
account.withdraw(1000);   // May trigger ERROR (insufficient funds)

[11:40:45.847] [INFO ] [BankAccount] Account created: RONI with initial balance INR 5000.0
[11:40:45.861] [DEBUG] [BankAccount] Deposit requested. Current balance=5000.0, amount=1000.0
[11:40:45.863] [DEBUG] [BankAccount] Balance updated after deposit: 6000.0
[11:40:45.865] [INFO ] [BankAccount] Deposit successful: INR 1000.0 | New balance: INR 6000.0
[11:40:45.875] [DEBUG] [BankAccount] Withdrawal requested. Current balance=6000.0, amount=2000.0
[11:40:45.876] [DEBUG] [BankAccount] Balance updated after withdrawal: 4000.0
[11:40:45.877] [INFO ] [BankAccount] Withdrawal successful: INR 2000.0 | Remaining balance: INR 4000.0
[11:40:45.886] [DEBUG] [BankAccount] Withdrawal requested. Current balance=4000.0, amount=3500.0
[11:40:45.887] [WARN ] [BankAccount] Suspicious activity: withdrawal amount INR 3500.0 is more than 80% of balance
[11:40:45.888] [DEBUG] [BankAccount] Balance updated after withdrawal: 500.0
[11:40:45.889] [INFO ] [BankAccount] Withdrawal successful: INR 3500.0 | Remain